## Imports and Configs

In [1]:
import sys
from pathlib import Path
#Setting the repo root so all of the folders and sub-folders are easily accessible
REPO_ROOT = Path.cwd().parent
sys.path.append(str(REPO_ROOT))


from experiments.helpers.ioi_dataset import IOIDataset #Main dataset class
import experiments.helpers.ioi_dataset as ioi_module #So we can access the nice variable lists easily

#Standard imports
import torch
import random
import numpy as np
from pathlib import Path
from huggingface_hub import login
from transformer_lens import HookedTransformer

N_PROMPTS = 5000

In [2]:
device = "cuda" #The SAE and model can't run on the CPU
#Setting up the dataset dir
DATA_DIR = REPO_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

#For reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

## Logging into HF

In [3]:
#Logging into hf so we can access the model weights
login()

## Loading in the model weights

In [4]:
#Loading in the Gemma model, note we are gonna use the base model coz the SAEs weren't trained on the instruct
#and we really only need the next token for IOI
model = HookedTransformer.from_pretrained_no_processing(
    "gemma-2-2b",
    device=device,
    dtype=torch.bfloat16,   # Gemma was trained in bf16 so no need for FP32
)
model.eval() #We do this so that the dropout layers don't have any effect
tokenizer = model.tokenizer #We will need this so we can change the padding token to right instead of left which IOI GPT 2 was designed for

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer


In [5]:
#What the tokenizer object has:
print(f"The Model is: {model.cfg.model_name}")
#The res stream that flows through, the layers are one attention(multi-head), one mlp
print(f"Dim of the res stream: {model.cfg.d_model}, n_layers: {model.cfg.n_layers}") 
print(f"BOS token: {tokenizer.bos_token!r} (id={tokenizer.bos_token_id})") #Basically SOS token
print(f"PAD token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
tokenizer.padding_side = "right" #We'll set this to right so then the first pad token we see will basically function as the end of output for us

The Model is: gemma-2-2b
Dim of the res stream: 2304, n_layers: 26
BOS token: '<bos>' (id=2)
PAD token: '<pad>' (id=0)


## Processing the names, places and objects

In [6]:
original_names = ioi_module.NAMES.copy()
original_places = ioi_module.PLACES.copy()
original_objs = ioi_module.OBJECTS.copy()

In [7]:
print(f"Here are some of the names: {original_names[:5]}, here are how many they are: {len(original_names)}")
print(f"Here are some of the places: {original_places[:5]}, here are how many they are: {len(original_places)}")
print(f"Here are some of the objects: {original_objs[:5]}, here are how many they are: {len(original_objs)}")

Here are some of the names: ['Michael', 'Christopher', 'Jessica', 'Matthew', 'Ashley'], here are how many they are: 99
Here are some of the places: ['store', 'garden', 'restaurant', 'school', 'hospital'], here are how many they are: 8
Here are some of the objects: ['ring', 'kiss', 'bone', 'basketball', 'computer'], here are how many they are: 8


In [8]:
#Next we tokenize all of these words, I've decided to do this so we can just check if any tokenize to more...
#...than one token. It's important we filter these out, if there are, as we only want to work with the next token prediction
def assert_single_token(words, label):
    multi = [w for w in words if len(tokenizer.encode(" " + w, add_special_tokens=False)) != 1]
    msg = f"{label}: {len(words) - len(multi)}/{len(words)} single-token. Multi: {multi}"
    if multi:
        raise AssertionError(msg)
    print(msg + (" ok " if not multi else " not ok "))
 
assert_single_token(ioi_module.NAMES,   "NAMES")
assert_single_token(ioi_module.PLACES,  "PLACES")
assert_single_token(ioi_module.OBJECTS, "OBJECTS")

NAMES: 99/99 single-token. Multi: [] ok 
PLACES: 8/8 single-token. Multi: [] ok 
OBJECTS: 8/8 single-token. Multi: [] ok 


## Generating the dataset

In [9]:
print(f"Generating clean IOI dataset (N={N_PROMPTS})")
 
clean_dataset = IOIDataset(
    prompt_type="mixed",# half ABBA, half BABA where A is the obj, B the subject, nice to have some diversity
    N=N_PROMPTS,
    tokenizer=tokenizer,
    prepend_bos=False,# Gemma will handle the BOS
    symmetric=False,
)

Generating clean IOI dataset (N=5000)


In [10]:
print(f"Generated {len(clean_dataset)} prompts")
print(f"Example sentence: {clean_dataset.sentences[0]}")
print(f"  IO/S: {clean_dataset.ioi_prompts[0]['IO']} / {clean_dataset.ioi_prompts[0]['S']}")
print(f"The shape of the dataset: {tuple(clean_dataset.toks.shape)}")

Generated 5000 prompts
Example sentence: Then, Matthew and Robert had a lot of fun at the school. Robert gave a ring to Matthew
  IO/S: Matthew / Robert
The shape of the dataset: (5000, 22)


In [11]:
bos_id = tokenizer.bos_token_id
first, second = clean_dataset.toks[0, 0].item(), clean_dataset.toks[0, 1].item()

print("\nFirst prompt token-by-token (first 8 tokens):")
for i, tok_id in enumerate(clean_dataset.toks[0].tolist()):
    print(f"  [{i}] id={tok_id}  {tokenizer.decode([tok_id])}")
 
print(f"IO token: id={clean_dataset.io_tokenIDs[0]}, "
      f"decoded={tokenizer.decode([clean_dataset.io_tokenIDs[0]])}")
print(f"S  token: id={clean_dataset.s_tokenIDs[0]}, "
      f"decoded={tokenizer.decode([clean_dataset.s_tokenIDs[0]])}")
print(f"End position: {clean_dataset.word_idx['end'][0].item()}")



First prompt token-by-token (first 8 tokens):
  [0] id=2  <bos>
  [1] id=7409  Then
  [2] id=235269  ,
  [3] id=23739   Matthew
  [4] id=578   and
  [5] id=7866   Robert
  [6] id=1093   had
  [7] id=476   a
  [8] id=2940   lot
  [9] id=576   of
  [10] id=2245   fun
  [11] id=696   at
  [12] id=573   the
  [13] id=2493   school
  [14] id=235265  .
  [15] id=7866   Robert
  [16] id=5645   gave
  [17] id=476   a
  [18] id=7408   ring
  [19] id=577   to
  [20] id=23739   Matthew
  [21] id=0  <pad>
IO token: id=23739, decoded= Matthew
S  token: id=7866, decoded= Robert
End position: 19


In [12]:
#Now we can generate the corrupted dataset where the Subject and the indirect obj are flipped
print("\nGenerating corrupted dataset...")
corrupted_dataset = clean_dataset.gen_flipped_prompts(("S", "IO"))

print(f"Clean[0]:     {clean_dataset.sentences[0]}")
print(f"Corrupted[0]: {corrupted_dataset.sentences[0]}")


Generating corrupted dataset...
Clean[0]:     Then, Matthew and Robert had a lot of fun at the school. Robert gave a ring to Matthew
Corrupted[0]: Then, Robert and Matthew had a lot of fun at the school. Matthew gave a ring to Robert


In [13]:
torch.save({
    "clean_toks":             clean_dataset.toks,
    "corrupted_toks":         corrupted_dataset.toks,
    "clean_io_token_ids":     torch.tensor(clean_dataset.io_tokenIDs),
    "clean_s_token_ids":      torch.tensor(clean_dataset.s_tokenIDs),
    "corrupted_io_token_ids": torch.tensor(corrupted_dataset.io_tokenIDs),
    "corrupted_s_token_ids":  torch.tensor(corrupted_dataset.s_tokenIDs),
    "end_positions":          clean_dataset.word_idx["end"],
    "word_idx":               clean_dataset.word_idx,
    "sentences":              clean_dataset.sentences,
    "corrupted_sentences":    corrupted_dataset.sentences,
    "N":                      N_PROMPTS,
    "seed":                   SEED,
}, DATA_DIR / "ioi_dataset.pt")
 
print(f"\nSaved to {DATA_DIR / 'ioi_dataset.pt'}")

NameError: name 'SEED' is not defined